# Toy Example: Method Walkthrough

A small, fully explicit run of the method on 6 models, 10 samples, and 3 classes
-- the worked example from the paper's appendix. It needs no datasets and no
training: it imports the actual method from `src/` and prints the disagreement
geometry, the Dunn-selected expert sets, and the resulting predictions.

Because it calls the shipped modules directly, it doubles as an end-to-end check
that `src/` reproduces the appendix: the expert sets below should match
E_1 = {1,2,3,5,6}, E_2 = {2,4,5,6}, E_3 = {2,5,6}.


In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), "src"))

import numpy as np
from disagreement import class_disagreement_matrix
from dunn import find_dunn_experts
from voting import experts_weighted_voting, majority_voting_experts, predictions_to_vector

## Toy data

True labels and the predicted labels of six models on ten samples (classes and
model ids are 1-indexed, matching the appendix). Models 1-3 are aligned on class
1; models 4-6 form a tighter consensus.

In [ ]:
y_true = [1, 2, 3, 1, 2, 3, 1, 2, 3, 3]  # 10 samples

predictions = {
    # Models 1-3
    1: [1, 3, 2, 1, 3, 2, 1, 3, 2, 2],
    2: [1, 2, 3, 1, 2, 3, 1, 2, 3, 3],
    3: [1, 2, 2, 1, 2, 2, 1, 3, 3, 2],
    # Models 4-6
    4: [3, 2, 1, 3, 2, 1, 3, 2, 1, 1],
    5: [1, 2, 3, 1, 2, 3, 1, 2, 3, 3],
    6: [1, 2, 3, 1, 2, 3, 1, 2, 3, 3],
}
C = 3

for mid, preds in predictions.items():
    assert len(preds) == 10 and all(p in (1, 2, 3) for p in preds)

## Per-class prediction sets

`results[c][m]` lists the sample indices model `m` assigns to class `c` -- the
representation the method consumes.

In [ ]:
results = {c: {} for c in range(1, C + 1)}
for c in range(1, C + 1):
    for mid in sorted(predictions):
        results[c][mid] = [i for i, lab in enumerate(predictions[mid]) if lab == c]

for c in range(1, C + 1):
    print(f"class {c}: " + ", ".join(f"m{m}->{results[c][m]}" for m in sorted(results[c])))

## Disagreement geometry (class 2)

The class-2 disagreement matrix. Models 2, 4, 5, 6 form a perfect-consensus block
(zero pairwise disagreement); model 3 is partially aligned; model 1 disagrees
strongly. This is the block structure the Dunn criterion exploits.

In [ ]:
D2 = class_disagreement_matrix(results[2])
order = sorted(results[2].keys())
print("      " + "  ".join(f"m{m}" for m in order))
for i, m in enumerate(order):
    print(f"m{m}: " + "  ".join(f"{D2[i, j]:.2f}" for j in range(len(order))))

## Expert identification and aggregation

Run the Dunn search and both aggregation rules. The expert sets should match the
appendix.

In [ ]:
dunn_partition, _, dunn_scores = find_dunn_experts(results, min_subset_size=3, max_subset_size=10)
print("Dunn expert sets:")
for c in sorted(dunn_partition):
    print(f"  E_{c} = {{{', '.join(str(m) for m in dunn_partition[c])}}}")

print("\nExpected (appendix): E_1 = {1, 2, 3, 5, 6}, E_2 = {2, 4, 5, 6}, E_3 = {2, 5, 6}")

weighted_final, voting_weights = experts_weighted_voting(dunn_partition, results)
majority_final = majority_voting_experts(dunn_partition, results)

print("\nExpert-weighted vote (class -> sample indices):")
for c in sorted(weighted_final):
    print(f"  class {c}: {sorted(weighted_final[c])}")
print("\nExpert-majority vote (class -> sample indices):")
for c in sorted(majority_final):
    print(f"  class {c}: {sorted(majority_final[c])}")

## Per-class model accuracies

Overall and per-class accuracy for each model, reproducing the appendix
accuracy table.


In [ ]:
import numpy as np
yt = np.array(y_true)

print(f"{'Model':6s} {'Overall':>8s} {'Class 1':>8s} {'Class 2':>8s} {'Class 3':>8s}")
for mid in sorted(predictions):
    yp = np.array(predictions[mid])
    overall = np.mean(yp == yt)
    per_class = []
    for c in range(1, C + 1):
        mask = yt == c
        per_class.append(np.mean(yp[mask] == c))
    print(f"m{mid:<5d} {overall:8.3f} " + " ".join(f"{v:8.3f}" for v in per_class))